# LongCat-Video-Avatar 1.5 저메모리 HTTP 서버 · Colab A100

이 노트북은 가족센터 앱과 자동 연결되지 않습니다. 전용 GPU가 있는 환경에서만 실행하고, 마지막 셀이 출력하는 값을 로컬 `.env`에 **수동으로 반영한 경우에만** 앱이 LongCat 워커를 호출합니다.

기존 A100 영상 사이드카가 담당하던 HTTP 아바타 서버 자리를 LongCat으로 교체하기 위한 노트북입니다. LongCat은 경량 립싱크 모델보다 훨씬 무거우므로 믿:음·OCR·TTS와 같은 GPU에서 함께 실행하지 않고, 별도의 Colab A100 런타임을 영상 생성 전용으로 사용합니다.

- API: `/v1/avatar/status`, `/v1/avatar/render`, `/v1/avatar/media/{filename}`
- 기본 생성: INT8, 순차 오프로딩, 480p, 동시 작업 1개
- 권장 GPU: A100 40GB 이상과 시스템 RAM 45GiB 이상
- Colab Secrets: `NGROK_AUTHTOKEN` 필수, `LONGCAT_WORKER_API_KEY` 선택
- Colab 탭과 런타임이 살아 있는 동안만 동작하며 재연결하면 URL이 바뀝니다.
- 실제 내담자 사진·음성은 사용하지 말고 합성 교육 페르소나만 사용하세요.


## 1. Drive, GPU, RAM 및 저장공간 확인


In [ ]:
import os, re, shutil, subprocess, sys, time, secrets
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'
]).decode().strip()
print('GPU:', gpu_line)
gpu_memory_mib = int(gpu_line.rsplit(',', 1)[1].strip())
if gpu_memory_mib < 38000:
    raise RuntimeError('최소 A100 40GB급 GPU가 필요합니다.')

with open('/proc/meminfo', encoding='utf-8') as meminfo:
    mem_total_kib = int(re.search(r'MemTotal:\s+(\d+)', meminfo.read()).group(1))
ram_gib = mem_total_kib / 1024**2
print(f'시스템 RAM: {ram_gib:.1f} GiB')
if ram_gib < 45:
    raise RuntimeError('저메모리 워커도 시스템 RAM 45GiB 이상이 필요합니다.')

free_gib = shutil.disk_usage('/content').free / 1024**3
print(f'/content 빈 공간: {free_gib:.1f} GiB')
if free_gib < 120:
    raise RuntimeError('/content에 최소 120GiB의 빈 공간이 필요합니다.')

ROOT = Path('/content/longcat_avatar15')
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('전용 런타임 검사 완료')


## 2. 공식 코드·전용 Python 3.10 환경·필요 가중치 준비
최초 실행에는 모델 다운로드 시간이 필요합니다. 모델은 `/content`에만 저장됩니다.


In [ ]:
# 2. 공식 코드와 전용 Python 3.10 환경 설치, 필요한 가중치만 다운로드
def run(command, *, cwd=None, env=None):
    print('\n$', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'git', 'git-lfs', 'libsndfile1'])
run(['git', 'lfs', 'install'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

REPO = ROOT / 'repo'
ENV_DIR = ROOT / '.venv'
PYTHON = ENV_DIR / 'bin' / 'python'
HF = ENV_DIR / 'bin' / 'hf'
WEIGHTS = ROOT / 'weights'
BASE_MODEL = WEIGHTS / 'LongCat-Video'
AVATAR_MODEL = WEIGHTS / 'LongCat-Video-Avatar-1.5'

if not REPO.exists():
    run(['git', 'clone', '--single-branch', '--branch', 'main',
         'https://github.com/meituan-longcat/LongCat-Video', str(REPO)])

if not PYTHON.exists():
    run(['uv', 'venv', '--python', '3.10', str(ENV_DIR)])

run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'torch==2.6.0+cu124', 'torchvision==0.21.0+cu124', 'torchaudio==2.6.0+cu124',
     '--index-url', 'https://download.pytorch.org/whl/cu124'])

requirements = (REPO / 'requirements.txt').read_text(encoding='utf-8').splitlines()
requirements = [line for line in requirements if line.strip()
                and not line.startswith('torch==')
                and not line.startswith('flash-attn==')]
filtered_requirements = ROOT / 'requirements_colab.txt'
filtered_requirements.write_text('\n'.join(requirements) + '\n', encoding='utf-8')

run(['uv', 'pip', 'install', '--python', str(PYTHON), '-r', str(filtered_requirements)])
avatar_requirements = (REPO / 'requirements_avatar.txt').read_text(encoding='utf-8').splitlines()
avatar_requirements = [line for line in avatar_requirements if line.strip()
                       and not line.startswith('libsndfile1==')
                       and not line.startswith('tritonserverclient==')
                       and not line.startswith('openai==')]
filtered_avatar_requirements = ROOT / 'requirements_avatar_colab.txt'
filtered_avatar_requirements.write_text('\n'.join(avatar_requirements) + '\n', encoding='utf-8')
run(['uv', 'pip', 'install', '--python', str(PYTHON), '-r', str(filtered_avatar_requirements)])
run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'huggingface_hub[cli]', 'ninja', 'packaging', 'wheel', 'setuptools'])

flash_env = os.environ.copy()
flash_env['MAX_JOBS'] = '4'
flash_env['TORCH_CUDA_ARCH_LIST'] = '8.0'
run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'flash-attn==2.7.4.post1', '--no-build-isolation'], env=flash_env)

model_env = os.environ.copy()
model_env['HF_HOME'] = '/content/hf_cache'
BASE_MODEL.mkdir(parents=True, exist_ok=True)
AVATAR_MODEL.mkdir(parents=True, exist_ok=True)

run([str(HF), 'download', 'meituan-longcat/LongCat-Video',
     '--include', 'tokenizer/*', 'text_encoder/*', 'vae/*', 'config.json', 'model_index.json',
     '--local-dir', str(BASE_MODEL)], env=model_env)
run([str(HF), 'download', 'meituan-longcat/LongCat-Video-Avatar-1.5',
     '--include', 'base_model_int8/*', 'lora/*', 'scheduler/*',
     'vocal_separator/*', 'whisper-large-v3/*', 'config.json', 'model_index.json',
     '--local-dir', str(AVATAR_MODEL)], env=model_env)

run([str(PYTHON), '-c',
     "import torch, flash_attn; print('torch', torch.__version__); print(torch.cuda.get_device_name(0))"])

# 공식 저장소 PR #115의 저메모리 단일 GPU 실행 파일만 가져온다.
LOWMEM_SCRIPT = REPO / 'run_demo_avatar_single_lowmem.py'
run(['git', 'fetch', 'origin', '+pull/115/head:lowmem-pr'], cwd=str(REPO))
lowmem_source = subprocess.check_output(
    ['git', 'show', 'lowmem-pr:run_demo_avatar_single_lowmem.py'], cwd=str(REPO)
)
LOWMEM_SCRIPT.write_bytes(lowmem_source)
print('\n설치와 모델 다운로드 완료')

## 3. 저메모리 연속 생성 패치 확인


In [ ]:
LOWMEM_SCRIPT = REPO / 'run_demo_avatar_single_lowmem.py'
lowmem_text = LOWMEM_SCRIPT.read_text(encoding='utf-8')
lowmem_text = lowmem_text.replace(
    'generator.manual_seed(42 + global_rank)',
    "generator.manual_seed(int(os.environ.get('LONGCAT_SEED', '42')) + global_rank)"
)
lowmem_text = lowmem_text.replace('offload_kv_cache=False', 'offload_kv_cache=True')
LOWMEM_SCRIPT.write_text(lowmem_text, encoding='utf-8')
print('시드 제어와 KV 캐시 CPU 오프로딩 적용 완료')


## 4. 가족센터 LongCat 워커 코드 배치
워커 코드는 이 노트북 생성 시 저장소의 `workers/longcat_avatar/app.py`에서 자동 포함됩니다.


In [ ]:
WORKER_PROJECT = Path('/content/family_center_longcat_worker')
PACKAGE_DIR = WORKER_PROJECT / 'workers' / 'longcat_avatar'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(WORKER_PROJECT / 'workers' / '__init__.py').write_text('', encoding='utf-8')
(PACKAGE_DIR / '__init__.py').write_text('', encoding='utf-8')
WORKER_SOURCE = 'from __future__ import annotations\n\nimport base64\nimport binascii\nimport json\nimport math\nimport os\nimport re\nimport subprocess\nimport threading\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import FileResponse\nfrom pydantic import BaseModel, Field\n\n\nMODEL_ID = "meituan-longcat/LongCat-Video-Avatar-1.5"\nROOT = Path(os.getenv("LONGCAT_WORKER_ROOT", "/content/longcat_avatar_worker"))\nREPO = Path(os.getenv("LONGCAT_REPO", "/content/longcat_avatar15/repo"))\nPYTHON = Path(os.getenv("LONGCAT_PYTHON", "/content/longcat_avatar15/.venv/bin/python"))\nCHECKPOINT = Path(\n    os.getenv(\n        "LONGCAT_CHECKPOINT",\n        "/content/longcat_avatar15/weights/LongCat-Video-Avatar-1.5",\n    )\n)\nLOWMEM_SCRIPT = Path(\n    os.getenv(\n        "LONGCAT_LOWMEM_SCRIPT",\n        str(REPO / "run_demo_avatar_single_lowmem.py"),\n    )\n)\nFFMPEG = os.getenv("FFMPEG_BIN", "ffmpeg")\nFFPROBE = os.getenv("FFPROBE_BIN", "ffprobe")\nRESOLUTION = os.getenv("LONGCAT_RESOLUTION", "480p")\nAPI_KEY = os.getenv("LONGCAT_WORKER_API_KEY", "")\nSEED = int(os.getenv("LONGCAT_SEED", "29411"))\nMAX_INPUT_MB = int(os.getenv("LONGCAT_MAX_INPUT_MB", "25"))\n\nMEDIA_DIR = ROOT / "media"\nJOB_DIR = ROOT / "jobs"\nMEDIA_DIR.mkdir(parents=True, exist_ok=True)\nJOB_DIR.mkdir(parents=True, exist_ok=True)\n\n# Low-memory inference is deliberately serialized. The web application remains\n# responsive because this process is deployed separately from LLM/TTS/STT.\nGPU_LOCK = threading.Lock()\n\napp = FastAPI(title="LongCat Avatar 1.5 GPU Worker", version="0.1.0")\n\n\nclass AvatarRenderRequest(BaseModel):\n    turn_id: str = Field(min_length=1, max_length=120)\n    text: str = Field(min_length=1, max_length=4000)\n    emotion: str = "neutral"\n    persona_id: str = "lee-jieun"\n    source_image_base64: str = Field(min_length=100)\n    source_image_mime_type: str = "image/png"\n    audio_url: str = Field(min_length=10)\n    cache_key: str | None = None\n\n\ndef _authorize(authorization: str | None) -> None:\n    if not API_KEY:\n        return\n    if authorization != f"Bearer {API_KEY}":\n        raise HTTPException(status_code=401, detail="Invalid worker API key")\n\n\ndef _safe_name(value: str | None, fallback: str = "turn") -> str:\n    normalized = re.sub(r"[^A-Za-z0-9_-]", "", value or "")[:120]\n    return normalized or fallback\n\n\ndef _decode_image(value: str) -> bytes:\n    encoded = value.split(",", 1)[1] if value.startswith("data:") else value\n    try:\n        raw = base64.b64decode(encoded, validate=True)\n    except (ValueError, binascii.Error) as exc:\n        raise HTTPException(status_code=422, detail="Invalid source image") from exc\n    if not raw or len(raw) > MAX_INPUT_MB * 1024 * 1024:\n        raise HTTPException(status_code=413, detail="Source image is empty or too large")\n    return raw\n\n\ndef _read_audio(value: str) -> bytes:\n    if value.startswith("data:"):\n        try:\n            return base64.b64decode(value.split(",", 1)[1], validate=True)\n        except (ValueError, binascii.Error) as exc:\n            raise HTTPException(status_code=422, detail="Invalid audio data URL") from exc\n    request = urllib.request.Request(\n        value,\n        headers={"Accept": "audio/*", "ngrok-skip-browser-warning": "1"},\n    )\n    with urllib.request.urlopen(request, timeout=180) as response:\n        raw = response.read(MAX_INPUT_MB * 1024 * 1024 + 1)\n    if not raw or len(raw) > MAX_INPUT_MB * 1024 * 1024:\n        raise HTTPException(status_code=413, detail="Audio is empty or too large")\n    return raw\n\n\ndef _duration_seconds(path: Path) -> float:\n    result = subprocess.run(\n        [\n            FFPROBE,\n            "-v",\n            "error",\n            "-show_entries",\n            "format=duration",\n            "-of",\n            "default=noprint_wrappers=1:nokey=1",\n            str(path),\n        ],\n        check=True,\n        capture_output=True,\n        text=True,\n    )\n    return float(result.stdout.strip())\n\n\ndef _num_segments(duration: float) -> int:\n    first_segment_seconds = 93 / 25\n    following_segment_seconds = (93 - 13) / 25\n    return max(\n        1,\n        1 + math.ceil(max(0.0, duration - first_segment_seconds) / following_segment_seconds),\n    )\n\n\ndef _prompt(emotion: str) -> str:\n    emotional_detail = {\n        "hurt": (\n            "She feels quietly hurt and disappointed, never angry. Hurt is shown by a "\n            "softened gaze, slight inner-eyebrow lift and subtle lip tension. Her eyebrows "\n            "never pull downward or strongly together; the glabella remains smooth."\n        ),\n        "sad": "She looks gently sad and reflective without crying, grimacing or melodrama.",\n        "anxious": "She looks mildly cautious, with calm breathing and no panic performance.",\n        "angry": "She is frustrated but socially restrained, never aggressive or threatening.",\n        "withdrawn": "She is reserved and quiet, with subdued eye contact and no dramatic motion.",\n        "neutral": "She remains calm, attentive and emotionally neutral.",\n    }.get(emotion, "She remains calm and emotionally restrained.")\n    return " ".join(\n        [\n            "Locked-off medium close-up in a quiet family counseling room.",\n            "A Korean adult client speaks softly to the counselor.",\n            emotional_detail,\n            "Accurate lip shapes, natural jaw and chin motion, and minimal lower-cheek support.",\n            "Two or three slow irregular blinks, tiny eye refocusing and quiet breathing.",\n            "The head is almost still and the crown, hairline, ears, neck and shoulders remain coherent.",\n            "The expression stays close to the source portrait and never accumulates across the shot.",\n            "No exaggerated acting, repeated motion, camera movement, face warping or identity drift.",\n        ]\n    )\n\n\ndef _assert_runtime_ready() -> None:\n    missing = [path for path in (REPO, PYTHON, CHECKPOINT, LOWMEM_SCRIPT) if not path.exists()]\n    if missing:\n        raise HTTPException(\n            status_code=503,\n            detail="LongCat worker is not provisioned: " + ", ".join(map(str, missing)),\n        )\n    script = LOWMEM_SCRIPT.read_text(encoding="utf-8")\n    patched = script.replace(\n        "generator.manual_seed(42 + global_rank)",\n        "generator.manual_seed(int(os.environ.get(\'LONGCAT_SEED\', \'42\')) + global_rank)",\n    ).replace("offload_kv_cache=False", "offload_kv_cache=True")\n    if patched != script:\n        LOWMEM_SCRIPT.write_text(patched, encoding="utf-8")\n\n\n@app.get("/v1/avatar/status")\ndef status(authorization: str | None = Header(default=None)) -> dict:\n    _authorize(authorization)\n    missing = [str(path) for path in (REPO, PYTHON, CHECKPOINT, LOWMEM_SCRIPT) if not path.exists()]\n    gpu = None\n    try:\n        gpu = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],\n            text=True,\n            timeout=5,\n        ).strip()\n    except (OSError, subprocess.SubprocessError):\n        pass\n    return {\n        "status": "ok" if not missing and gpu else "not_ready",\n        "provider": "longcat_avatar_15_lowmem",\n        "model": MODEL_ID,\n        "resolution": RESOLUTION,\n        "gpu": gpu,\n        "missing": missing,\n        "concurrency": 1,\n    }\n\n\n@app.post("/v1/avatar/render")\ndef render(\n    payload: AvatarRenderRequest,\n    authorization: str | None = Header(default=None),\n) -> dict:\n    _authorize(authorization)\n    _assert_runtime_ready()\n    safe_id = _safe_name(payload.cache_key or payload.turn_id)\n    final_path = MEDIA_DIR / f"{safe_id}.mp4"\n    if final_path.exists() and final_path.stat().st_size > 100_000:\n        return {\n            "provider": "longcat_avatar_15_lowmem",\n            "model": MODEL_ID,\n            "video_url": f"/v1/avatar/media/{final_path.name}",\n            "cached": True,\n            "render_ms": 0,\n        }\n\n    started = time.perf_counter()\n    work = JOB_DIR / f"{safe_id}-{int(started * 1000)}"\n    work.mkdir(parents=True, exist_ok=False)\n    source_image = work / "source.png"\n    source_audio = work / "speech_input"\n    normalized_audio = work / "speech.wav"\n    source_image.write_bytes(_decode_image(payload.source_image_base64))\n    source_audio.write_bytes(_read_audio(payload.audio_url))\n\n    subprocess.run(\n        [\n            FFMPEG,\n            "-y",\n            "-v",\n            "warning",\n            "-i",\n            str(source_audio),\n            "-ar",\n            "16000",\n            "-ac",\n            "1",\n            "-c:a",\n            "pcm_s16le",\n            str(normalized_audio),\n        ],\n        check=True,\n    )\n    duration = _duration_seconds(normalized_audio)\n    segments = _num_segments(duration)\n    job_json = work / "job.json"\n    job_json.write_text(\n        json.dumps(\n            {\n                "prompt": _prompt(payload.emotion),\n                "cond_image": str(source_image),\n                "cond_audio": {"person1": str(normalized_audio)},\n            },\n            ensure_ascii=False,\n            indent=2,\n        ),\n        encoding="utf-8",\n    )\n    generated_dir = work / "generated"\n    generated_dir.mkdir()\n    command = [\n        str(PYTHON),\n        "-m",\n        "torch.distributed.run",\n        "--nproc_per_node=1",\n        str(LOWMEM_SCRIPT),\n        "--checkpoint_dir",\n        str(CHECKPOINT),\n        "--stage_1",\n        "ai2v",\n        "--input_json",\n        str(job_json),\n        "--resolution",\n        RESOLUTION,\n        "--num_segments",\n        str(segments),\n        "--output_dir",\n        str(generated_dir),\n        "--ref_img_index",\n        "0",\n        "--mask_frame_range",\n        "3",\n    ]\n    environment = os.environ.copy()\n    environment["PYTHONPATH"] = str(REPO)\n    environment["LONGCAT_SEED"] = str(SEED)\n    environment["PYTHONUNBUFFERED"] = "1"\n    log_path = work / "render.log"\n\n    with GPU_LOCK, log_path.open("w", encoding="utf-8") as log:\n        result = subprocess.run(\n            command,\n            cwd=str(REPO),\n            env=environment,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            text=True,\n            check=False,\n        )\n    if result.returncode != 0:\n        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]\n        raise HTTPException(status_code=500, detail="LongCat render failed: " + " | ".join(tail))\n\n    generated = generated_dir / ("final_video.mp4" if segments > 1 else "segment_001.mp4")\n    if not generated.exists():\n        raise HTTPException(status_code=500, detail="LongCat did not create the expected MP4")\n\n    temporary = MEDIA_DIR / f".{safe_id}.part.mp4"\n    # Explicit stream mapping avoids selecting the per-segment temporary audio.\n    subprocess.run(\n        [\n            FFMPEG,\n            "-y",\n            "-v",\n            "warning",\n            "-i",\n            str(generated),\n            "-i",\n            str(normalized_audio),\n            "-map",\n            "0:v:0",\n            "-map",\n            "1:a:0",\n            "-c:v",\n            "copy",\n            "-c:a",\n            "aac",\n            "-shortest",\n            str(temporary),\n        ],\n        check=True,\n    )\n    temporary.replace(final_path)\n    return {\n        "provider": "longcat_avatar_15_lowmem",\n        "model": MODEL_ID,\n        "video_url": f"/v1/avatar/media/{final_path.name}",\n        "cached": False,\n        "render_ms": round((time.perf_counter() - started) * 1000),\n    }\n\n\n@app.get("/v1/avatar/media/{filename}")\ndef media(filename: str, authorization: str | None = Header(default=None)) -> FileResponse:\n    _authorize(authorization)\n    if Path(filename).name != filename or not filename.endswith(".mp4"):\n        raise HTTPException(status_code=404, detail="Video not found")\n    path = MEDIA_DIR / filename\n    if not path.exists():\n        raise HTTPException(status_code=404, detail="Video not found")\n    return FileResponse(path, media_type="video/mp4", filename=filename)\n'
WORKER_REQUIREMENTS = 'fastapi>=0.115,<1\nuvicorn[standard]>=0.30,<1\npydantic>=2.9,<3\n\n'
(PACKAGE_DIR / 'app.py').write_text(WORKER_SOURCE, encoding='utf-8')
(PACKAGE_DIR / 'requirements.txt').write_text(WORKER_REQUIREMENTS, encoding='utf-8')

subprocess.run([
    'uv', 'pip', 'install', '--python', str(PYTHON),
    '-r', str(PACKAGE_DIR / 'requirements.txt')
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok>=7,<8', 'requests>=2.32,<3'], check=True)
print('워커 코드와 서버 패키지 준비 완료:', WORKER_PROJECT)


## 5. Uvicorn 시작 및 로컬 상태 검증
이 셀은 워커만 시작하며 가족센터 앱 설정은 변경하지 않습니다.


In [ ]:
import requests

SERVER_PORT = 8015
try:
    WORKER_API_KEY = userdata.get('LONGCAT_WORKER_API_KEY')
except Exception:
    WORKER_API_KEY = None
WORKER_API_KEY = WORKER_API_KEY or secrets.token_urlsafe(32)

server_env = os.environ.copy()
server_env.update({
    'PYTHONPATH': str(WORKER_PROJECT),
    'LONGCAT_WORKER_ROOT': '/content/longcat_avatar_worker',
    'LONGCAT_REPO': str(REPO),
    'LONGCAT_PYTHON': str(PYTHON),
    'LONGCAT_CHECKPOINT': str(AVATAR_MODEL),
    'LONGCAT_LOWMEM_SCRIPT': str(LOWMEM_SCRIPT),
    'LONGCAT_RESOLUTION': '480p',
    'LONGCAT_WORKER_API_KEY': WORKER_API_KEY,
    'LONGCAT_SEED': '29411',
})

if 'longcat_server_process' in globals() and longcat_server_process.poll() is None:
    longcat_server_process.terminate()
    longcat_server_process.wait(timeout=15)
if 'longcat_server_log_handle' in globals() and not longcat_server_log_handle.closed:
    longcat_server_log_handle.close()

server_log_path = Path('/content/longcat_avatar_worker/server.log')
server_log_path.parent.mkdir(parents=True, exist_ok=True)
longcat_server_log_handle = server_log_path.open('a', encoding='utf-8')
longcat_server_process = subprocess.Popen([
    str(PYTHON), '-m', 'uvicorn', 'workers.longcat_avatar.app:app',
    '--host', '127.0.0.1', '--port', str(SERVER_PORT), '--log-level', 'info'
], cwd=str(WORKER_PROJECT), env=server_env, stdout=longcat_server_log_handle, stderr=subprocess.STDOUT)

headers = {'Authorization': f'Bearer {WORKER_API_KEY}'}
deadline = time.time() + 45
local_status = None
while time.time() < deadline:
    if longcat_server_process.poll() is not None:
        break
    try:
        response = requests.get(f'http://127.0.0.1:{SERVER_PORT}/v1/avatar/status', headers=headers, timeout=3)
        if response.ok:
            local_status = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not local_status:
    tail = server_log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-50:]
    raise RuntimeError('LongCat 로컬 서버 시작 실패:\n' + '\n'.join(tail))
if local_status.get('status') != 'ok':
    raise RuntimeError(f'LongCat 워커 준비 미완료: {local_status}')
print('로컬 LongCat API 정상:', local_status)


## 6. ngrok HTTPS 공개 및 외부 상태 검증
Colab Secrets의 `NGROK_AUTHTOKEN`이 필요합니다. 출력값을 복사하기 전까지 로컬 앱에는 연결되지 않습니다.


In [ ]:
from pyngrok import ngrok

try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK_AUTHTOKEN = None
if not NGROK_AUTHTOKEN:
    raise RuntimeError('Colab Secrets에 NGROK_AUTHTOKEN을 추가하고 노트북 액세스를 허용하세요.')

ngrok.set_auth_token(NGROK_AUTHTOKEN)
try:
    if 'longcat_tunnel' in globals():
        ngrok.disconnect(longcat_tunnel.public_url)
except Exception:
    pass
longcat_tunnel = ngrok.connect(SERVER_PORT, 'http')
PUBLIC_URL = longcat_tunnel.public_url.replace('http://', 'https://').rstrip('/')

public_headers = {
    'Authorization': f'Bearer {WORKER_API_KEY}',
    'ngrok-skip-browser-warning': '1',
}
external = requests.get(f'{PUBLIC_URL}/v1/avatar/status', headers=public_headers, timeout=30)
external.raise_for_status()
external_body = external.json()
if external_body.get('status') != 'ok':
    raise RuntimeError(f'외부 LongCat 상태가 정상이 아닙니다: {external_body}')

print('\n===== 전용 GPU 워커를 실제 연결할 때만 .env에 넣을 값 =====')
print('AVATAR_PROVIDER=longcat_http')
print(f'LONGCAT_AVATAR_BASE_URL={PUBLIC_URL}')
print(f'LONGCAT_AVATAR_API_KEY={WORKER_API_KEY}')
print('LONGCAT_AVATAR_REQUEST_TIMEOUT=7200')
print('\n외부 상태 검증 완료:', external_body)
print('이 값을 반영하지 않으면 가족센터는 계속 static_2d이며 GPU 워커를 호출하지 않습니다.')


## 7. 연결·종료 기준

1. 실제 연결 시험 때만 6번 셀의 네 줄을 로컬 `.env`에 반영합니다.
2. FastAPI를 재시작한 뒤 `/api/v1/avatar/status`에서 `provider=longcat_http`, `reachable=true`를 확인합니다.
3. LongCat은 가벼운 실시간 립싱크 모델이 아닙니다. 첫 시연 응답은 사전 제작 MP4를 계속 사용하고, 이 서버는 고품질 후속 영상을 비동기 또는 사전 생성할 때 사용합니다.
4. 연결 시험이 끝나면 로컬 `.env`를 `AVATAR_PROVIDER=static_2d`로 되돌립니다.
5. Colab 런타임을 종료하면 Uvicorn·ngrok·GPU 모델이 모두 내려갑니다.
